<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 03. Métricas de Evaluación — La prueba médica y la alarma contra incendios
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 08
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/08%20-%20Classification/Para%20Dummies/03_Evaluacion_de_Modelos_y_Metricas_Clasificacion_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

El "accuracy" (exactitud) puede mentir. Este cuaderno te enseña las métricas reales para evaluar un clasificador:

1. Por qué el accuracy **miente** cuando los datos están desbalanceados.
2. La **Matriz de Confusión** — las 4 celdas que lo explican todo.
3. **Precision, Recall y F1** — con analogías de alarmas de incendio.
4. La **Curva ROC** — la gráfica que muestra qué tan bueno es el modelo en general.

---
## 1. Por qué el Accuracy puede mentirte 🤥

**Escenario:** Tienes un detector de fraude bancario. De cada 1000 transacciones, solo 10 son fraude.

Un modelo que siempre diga "No es fraude" tendrá un accuracy del **99%** — ¡sin detectar ni un solo fraude!

```
  Modelo inútil:                    Modelo útil:
  "Siempre NO fraude"               Detecta fraudes correctamente
  Accuracy = 99%  ✅                Accuracy = 94% ✅
  Fraudes detectados = 0  ❌        Fraudes detectados = 8/10  ✅
```

> 💡 **Regla de oro:** Cuando las clases están desbalanceadas (99% de una, 1% de otra), el accuracy es inútil. Usa **Recall, Precision y F1**.

In [ ]:
import os, urllib.parse, urllib.request, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, roc_auc_score, roc_curve)

def load_dataset(filename, module_name="08 - Classification"):
    candidates = [f"data/{filename}", f"../{module_name}/data/{filename}", f"{module_name}/data/{filename}", filename]
    for path in candidates:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    target_path = f"data/{filename}"
    url = f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/{urllib.parse.quote(module_name)}/data/{urllib.parse.quote(filename)}"
    urllib.request.urlretrieve(url, target_path)
    return target_path

df = pd.read_csv(load_dataset('customer_churn.csv'))
target_col = df.columns[-1]
print(f"Dataset Churn: {df.shape[0]} clientes")
print(df[target_col].value_counts().rename({0:'No cancela (0)', 1:'Cancela (1)  '}))
print(f"\n⚠️ Solo el {df[target_col].mean():.1%} de los clientes cancela el servicio — desbalanceado!")

In [ ]:
X = df.drop(columns=[target_col]).select_dtypes('number')
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train_sc, y_train)
y_pred = modelo.predict(X_test_sc)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.1%}  ← ¿es esto suficiente para confiar en el modelo?")

---
## 2. La Matriz de Confusión — Las 4 tipos de predicciones 📊

La **Matriz de Confusión** divide las predicciones en 4 tipos:

| | El modelo dijo NO | El modelo dijo SÍ |
|---|:---:|:---:|
| **En realidad era NO** | ✅ **Verdadero Negativo (TN)** | ❌ **Falso Positivo (FP)** — Falsa alarma |
| **En realidad era SÍ** | ❌ **Falso Negativo (FN)** — Se escapó | ✅ **Verdadero Positivo (TP)** |

**Analogía con la alarma contra incendios:**
- 🔴 **FP** (Falso Positivo) = La alarma suena pero no hay incendio → Molestia innecesaria.
- 🚨 **FN** (Falso Negativo) = Hay incendio pero la alarma no suena → ¡Desastre!

In [ ]:
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap de la matriz
labels = np.array([[f'✅ TN\n{TN}\n(No cancela, bien)', f'❌ FP\n{FP}\n(Falsa alarma)'],
                   [f'🚨 FN\n{FN}\n(Se escapó!)', f'✅ TP\n{TP}\n(Detectado)']])
sns.heatmap(cm, annot=labels, fmt='', cmap='YlOrBr', ax=axes[0],
            linewidths=2, linecolor='white', cbar=False,
            xticklabels=['Pred. No cancela', 'Pred. Cancela'],
            yticklabels=['Real No cancela', 'Real Cancela'])
axes[0].set_title('📊 Matriz de Confusión', fontweight='bold', fontsize=13)

# Gráfico de proporciones
categorias = ['TN\n(Correcto\nNo cancela)', 'FP\n(Falsa\nalarma)', 'FN\n(Se\nescapó)', 'TP\n(Correcto\nCancela)']
valores = [TN, FP, FN, TP]
colores = ['#10b981', '#f97316', '#ef4444', '#6366f1']
bars = axes[1].bar(categorias, valores, color=colores, edgecolor='white', linewidth=2)
for bar, val in zip(bars, valores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(val), ha='center', fontweight='bold', fontsize=12)
axes[1].set_title('Conteo de cada tipo de predicción', fontweight='bold')
axes[1].set_ylabel('Número de clientes')

plt.suptitle('Evaluación del clasificador de Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Precision, Recall y F1 explicados con analogías 🏥

In [ ]:
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("📐 MÉTRICAS CLAVE:\n")
print(f"  ACCURACY  = {accuracy_score(y_test, y_pred):.1%}")
print(f"   ↳ De todos los clientes, ¿qué % clasifiqué bien?")
print(f"   ↳ ⚠️  Puede engañar si los datos están desbalanceados.\n")

print(f"  PRECISION = {precision:.1%}")
print(f"   ↳ De los clientes que dije 'va a cancelar', ¿cuántos realmente cancelaron?")
print(f"   ↳ Analogía: ¿qué tan confiable es cuando suena la alarma?\n")

print(f"  RECALL    = {recall:.1%}")
print(f"   ↳ De todos los que realmente cancelaron, ¿cuántos detecté?")
print(f"   ↳ Analogía: ¿cuántos incendios detectó la alarma? (lo más crítico en medicina)\n")

print(f"  F1-SCORE  = {f1:.1%}")
print(f"   ↳ Equilibrio entre Precision y Recall. Un único número que resume ambos.")
print(f"   ↳ Úsalo cuando necesitas balancear falsas alarmas vs casos perdidos.")

---
## 4. La Curva ROC — ¿Qué tan bueno es el modelo en general? 📈

La **Curva ROC** muestra cómo cambia la relación entre **detectar los positivos** y **generar falsas alarmas** al variar el umbral de decisión.

- Un modelo **perfecto** tiene ROC-AUC = 1.0 (detecta todos los positivos sin ninguna falsa alarma).
- Un modelo **al azar** tiene ROC-AUC = 0.5 (la línea diagonal).
- Nuestro modelo debería estar entre 0.5 y 1.0 — cuanto más cerca de 1.0, mejor.

In [ ]:
y_probs = modelo.predict_proba(X_test_sc)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
auc = roc_auc_score(y_test, y_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#f59e0b', linewidth=3, label=f'Modelo (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Modelo al azar (AUC = 0.500)')
plt.fill_between(fpr, tpr, alpha=0.15, color='#f59e0b')
plt.xlabel('Tasa de Falsas Alarmas\n(FPR = Falsos Positivos / Total Negativos Reales)', fontsize=11)
plt.ylabel('Tasa de Detección Real\n(TPR = Recall = Verdaderos Positivos / Total Positivos Reales)', fontsize=11)
plt.title('🎯 Curva ROC — Evaluación global del clasificador de Churn\n(más área bajo la curva = mejor modelo)',
          fontweight='bold', fontsize=12)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAUC = {auc:.3f}")
if auc >= 0.9: print("   → Excelente")
elif auc >= 0.8: print("   → Bueno")
elif auc >= 0.7: print("   → Aceptable")
else: print("   → Requiere mejora")

---
## 5. Resumen — ¿Qué métrica usar? 🎓

| Situación | Métrica recomendada |
|---|---|
| Datos balanceados, todos los errores igual de graves | Accuracy |
| Falsos positivos muy costosos (ej. spam mal marcado) | **Precision** |
| Falsos negativos muy costosos (ej. enfermedad no detectada) | **Recall** |
| Equilibrio general entre precisión y detección | **F1-Score** |
| Comparar modelos independientemente del umbral | **ROC-AUC** |

> 🚀 **Siguiente paso:** Ve al cuaderno `04_KNN_Clasificacion_y_Seleccion_Modelos_Dummies.ipynb`.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>